# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is defined via a Croissant schema JSON-LD:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and columns. All record sets, fields, and columns are referenced by their Croissant `@id` values.

We enumerate the record sets, and for each, their fields and columns.

In [ ]:
# List all record sets and their field/column IDs
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}  |  name: {record_set.name}")
    fields = getattr(record_set, 'fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - Field @id: {field.id}  |  name: {getattr(field, 'name', '')}")
    columns = getattr(record_set, 'columns', [])
    if columns:
        print("  Columns:")
        for column in columns:
            print(f"    - Column @id: {column.id}  |  name: {getattr(column, 'name', '')}")
    print("")

## 3. Data Extraction
Load data from specific record set(s) into Pandas DataFrames using their `@id`s. 

For analysis in this notebook, select the main record set (usually with protein data table).

In [ ]:
#--- Find the main record set for protein analysis (example: using @id -- adjust as needed) --
# Let's list the record set @id again for reference
record_sets = metadata.record_sets

# If you know the main data table, use its @id, else pick the first one
main_record_set_id = None
for rs in record_sets:
    # Heuristically select the main record set, e.g. by name or id
    if rs.name and 'protein' in rs.name.lower():
        main_record_set_id = rs.id
        break
if not main_record_set_id and record_sets:
    main_record_set_id = record_sets[0].id

print(f"Selected RecordSet for DataFrame: {main_record_set_id}")

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df

if main_record_set_id in dataframes:
    print(f"Columns in '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print(f"No records found for RecordSet {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Common data processing: filtering, normalization, grouping. 

Below we:
- select a numeric field (e.g., "Molecular Weight (MW)", "Abundance", etc. -- adjust field `@id` to match schema),
- filter records,
- normalize values,
- and group by a categorical field if available.

In [ ]:
# --- Define field/column ids for numeric and grouping attributes ---
# Change these to the actual Field/@id in your schema.
numeric_field_id = None
group_field_id = None

# Inspect available columns
df = dataframes.get(main_record_set_id)
print("Columns:", df.columns.tolist())
# For example, let's try to select 'molecular_weight', 'normalized_abundance', or similar
for col in df.columns:
    if "weight" in col.lower():
        numeric_field_id = col
    elif "abundance" in col.lower() and not numeric_field_id:
        numeric_field_id = col
    elif "type" in col.lower() or 'group' in col.lower() or col.lower()=='category':
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Using group field: {group_field_id}")

# Filter and normalize
if numeric_field_id:
    # Filter out non-numeric values and missing values
    filtered_df = df.copy()
    filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').notnull()]
    filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)
    threshold = filtered_df[numeric_field_id].mean() # Example: mean threshold
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA. Please check the record set column names.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and relationship to groups (if available).

We'll use standard matplotlib and seaborn for quick plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Optional: Boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df.dropna(subset=[numeric_field_id, group_field_id]), x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, you loaded, explored, and visualized the FAIR<sup>2</sup> mass spectrometry dataset of extracellular vesicle proteins using the `mlcroissant` library. We walked through the Croissant schema, inspected record set structure, created pandas DataFrames, performed filtering and normalization, and visualized protein variables.

- **Data structure:** explored using Croissant `@id`s for traceable provenance
- **Numeric summaries:** identified and filtered high-abundance or high MW proteins
- **Visualization:** rapid overviews of the main quantitative fields

Continue with domain-specific analysis by customizing field IDs and operations as appropriate for your research.

---

_Notebook generated via reproducible Croissant schema from FAIR<sup>2</sup> project._